# TESSERA encoder: input contract & forward

**What it is:** a purpose-built agricultural Sentinel-1 + Sentinel-2 pixel-timeseries SSL model
(Cambridge, ucam-eo). The weights are open (CC0).

**Role here:** `src/models/tessera.py` runs the published model on each sample's time series:
`TesseraEncoder.encode(bench) -> (N, 128)`.

## Input contract: a time series, run through the model

TESSERA takes the S2 + S1 series directly (dual transformers, concat, then a linear reducer to 128).
Because the embedding is computed FROM the input, sensor-off / temporal-drop corruptions change it:
TESSERA is **condition-sensitive and stress-testable**, like Presto / OlmoEarth.

| input | from | notes |
|---|---|---|
| S2 | `bench.s2` (10 of B2..B12) | normalized with TESSERA's band stats |
| S1 | `bench.s1` (VV, VH) | see the S1-scale caveat below |
| DOY | `bench.doy` | real day-of-year where the dataset has it |
| **output** | `(N, 128)` | dim_reducer output of the dual-transformer fusion |

In [ ]:
# Bootstrap: find repo root and load a model module by path (src has no __init__ files).
import sys, importlib.util
from pathlib import Path
from types import SimpleNamespace
import numpy as np

REPO = Path.cwd()
while not (REPO / 'src').exists():
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

def load(name, rel):
    spec = importlib.util.spec_from_file_location(name, REPO / rel)
    m = importlib.util.module_from_spec(spec); sys.modules[name] = m; spec.loader.exec_module(m)
    return m

In [ ]:
tessera = load('tessera_mod', 'src/models/tessera.py')
print('embedding dim :', tessera.TESSERA_EMBEDDING_DIM)
print('S2 bands      :', tessera.TESSERA_S2_BANDS)
print('S1 bands      :', tessera.TESSERA_S1_BANDS)
print('weights path  :', tessera.DEFAULT_WEIGHTS, '(exists:', tessera.DEFAULT_WEIGHTS.exists(), ')')

In [ ]:
# A tiny synthetic Benchmark (duck-typed): just the attributes the encoder reads.
# Matches what src/dataio/get_input.py produces, but needs no data on disk.
N, T = 4, 12
rng = np.random.default_rng(0)
bench = SimpleNamespace(
    s2=(rng.random((N, T, 11)) * 5000).astype('float32'),   # B2..B12, NDVI (reflectance-ish)
    s1=(rng.random((N, T, 2)) * -15).astype('float32'),     # VV, VH (dB-ish)
    doy=np.tile(np.linspace(15, 350, T), (N, 1)).astype('float32'),
    s2_bands=['B2','B3','B4','B5','B6','B7','B8','B8A','B11','B12','NDVI'],
    s1_bands=['VV','VH'],
    n_samples=N,
)
print('synthetic bench:', N, 'samples x', T, 'timesteps')

## How one datapoint becomes model input

The wrapper selects TESSERA's 10 S2 bands (and VV/VH), normalizes with the published band stats,
and appends DOY as the last channel. Result per modality: `(N, T, bands+1)`.

In [ ]:
enc = tessera.TesseraEncoder(device='cpu', batch_size=4)
s2_in = enc._modality_input(bench.s2, tessera.TESSERA_S2_BANDS, bench.s2_bands,
                            tessera.S2_BAND_MEAN, tessera.S2_BAND_STD, bench.doy)
print('S2 model input:', s2_in.shape, '(N, T, 10 bands + DOY)')
print('one sample, first 3 steps:')
print(np.round(s2_in[0, :3], 2))

In [ ]:
# Single datapoint: the normalized 10-band S2 series TESSERA sees for field 0 (DOY column dropped).
import matplotlib.pyplot as plt
plt.figure(figsize=(9, 4))
for b, name in enumerate(tessera.TESSERA_S2_BANDS):
    plt.plot(bench.doy[0], s2_in[0, :, b], marker='.', label=name)
plt.title('TESSERA input: one field, 10 normalized S2 bands over the year')
plt.xlabel('day of year'); plt.ylabel('normalized reflectance'); plt.legend(ncol=5, fontsize=8)
plt.tight_layout(); plt.show()

## Forward pass: embedding (needs the TESSERA checkpoint)

Loads the published model and runs the real forward to `(N, 128)`. Zeroing S2 (sensor-off) changes
the embedding, which is the whole point: TESSERA responds to stress.

In [ ]:
try:
    emb = enc.encode(bench)
    print('embedding:', emb.shape, '| finite:', bool(np.isfinite(emb).all()))
    off = SimpleNamespace(**{**bench.__dict__})
    off.s2 = np.zeros_like(bench.s2)   # 'S2 off'
    delta = float(np.abs(emb - enc.encode(off)).max())
    print('condition-sensitive (S2 off): max delta', round(delta, 3))
except FileNotFoundError as e:
    print('Skipped real forward:', e)
    print('Download the TESSERA checkpoint and set TESSERA_WEIGHTS (or place it at the path above).')

## Caveats

: S1 scale. TESSERA trained on its own S1 scale (band means ~[5484, 3003]); our benchmark S1 is in
  dB. The S2 path matches; read S1-driven numbers with that caveat (see `src/models/tessera.py`).
: DOY. We use `bench.doy` (real where the dataset has it; CropHarvest's is synthetic monthly).
: Weights. The checkpoint is not redistributed; set `TESSERA_WEIGHTS` or drop it under `<cache>/tessera/`.